# Week 5 · Q12 — Why do 14 product–branch pairs differ from the ledger?
**Owner:** Ibrahim Khalil Umar (Data Analyst) · **Question:** 14 of 180 product–branch pairs differ between `inventory_master.current_stock` and the stock ledger (13 negative). Is the cause the unsigned `ADJUSTMENT` quantities?

Data: the corrected Week 2 merged datasets (`inventory_with_ledger.csv`, `stock_ledger_enriched.csv`, `purchase_orders_full.csv`).
The notebook searches its own folder and every parent for `week-02/reports/week_02/merged`, or use the `CADETX_MERGED` environment variable.
It writes outputs, charts and `validation_report_week05_q12.csv` next to itself and asserts that every check passes.

In [1]:
import os, pathlib
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = pathlib.Path.cwd()
def find_merged():
    env = os.environ.get("CADETX_MERGED")
    if env and pathlib.Path(env).exists():
        return pathlib.Path(env)
    for p in [HERE, *HERE.parents]:
        c = p / "week-02" / "reports" / "week_02" / "merged"
        if c.exists():
            return c
    raise SystemExit("Merged datasets not found: set CADETX_MERGED or run inside the repository checkout.")

MERGED = find_merged()
OUT = HERE / "q12_outputs"; CH = HERE / "q12_charts"
OUT.mkdir(exist_ok=True); CH.mkdir(exist_ok=True)
CHECKS = []
def check(name, passed, detail=""):
    CHECKS.append({"check": name, "passed": bool(passed), "detail": detail})
    print(("PASS " if passed else "FAIL ") + name + (" | " + str(detail) if detail != "" else ""))

K = ["product_id", "branch_id"]
L = pd.read_csv(MERGED / "stock_ledger_enriched.csv", low_memory=False, parse_dates=["movement_date"])
I = pd.read_csv(MERGED / "inventory_with_ledger.csv")
print("data folder:", MERGED)
print("ledger rows:", len(L), "| inventory pairs:", len(I))

data folder: <repository>/week-02/reports/week_02/merged
ledger rows: 237230 | inventory pairs: 180


## 1. Re-derive the 14 mismatches

In [2]:
check("180 product–branch pairs in inventory_master", len(I) == 180, len(I))
L["file_order"] = np.arange(len(L))
L = L.sort_values(K + ["file_order"]).reset_index(drop=True)
last_bal = L.groupby(K).running_balance.last().rename("last_running_balance").reset_index()
I = I.merge(last_bal, on=K)
check("inventory_with_ledger.ledger_latest_balance equals the last running_balance of every pair",
      (I.ledger_latest_balance == I.last_running_balance).all())
I["gap"] = I.current_stock - I.ledger_latest_balance
mism = I[I.gap != 0].copy()
check("14 pairs differ (reproduces Week 2/3)", len(mism) == 14, len(mism))
check("13 of the 14 gaps are negative", (mism.gap < 0).sum() == 13, f"{(mism.gap < 0).sum()} negative, {(mism.gap > 0).sum()} positive")
print(mism[K + ["current_stock", "ledger_latest_balance", "gap"]].to_string(index=False))
check("ledger rows are in chronological order within every pair (file order usable)",
      L.groupby(K).movement_date.apply(lambda s: s.is_monotonic_increasing).all())
check("no duplicate movement_id", not L.movement_id.duplicated().any())

PASS 180 product–branch pairs in inventory_master | 180
PASS inventory_with_ledger.ledger_latest_balance equals the last running_balance of every pair
PASS 14 pairs differ (reproduces Week 2/3) | 14
PASS 13 of the 14 gaps are negative | 13 negative, 1 positive
product_id branch_id  current_stock  ledger_latest_balance  gap
      P002    CHN001         105885                 106182 -297
      P004    PUN001         104859                 105024 -165
      P008    AHM001         109281                 109317  -36
      P010    AHM001         115149                 115549 -400
      P021    DEL001         115902                 116107 -205
      P023    HYD001         105798                 105981 -183
      P025    HYD001         113613                 114128 -515
      P026    PUN001         105190                 105335 -145
      P026    HYD001         108348                 108546 -198
      P028    DEL001         106744                 106967 -223
      P028    PUN001         105401

## 2. Is the `ADJUSTMENT` quantity unsigned?
Every `ADJUSTMENT` quantity is positive (5 to 50). The ledger also carries a `running_balance`, so the *applied* sign of each movement can be read from the step in the balance.

In [3]:
L = L.merge(I[K + ["opening_stock"]], on=K)
L["prev_balance"] = L.groupby(K).running_balance.shift()
first = L.groupby(K).cumcount() == 0
L["prev_balance"] = np.where(first, L.opening_stock, L.prev_balance)
L["step"] = L.running_balance - L.prev_balance

adj = L[L.movement_type == "ADJUSTMENT"].copy()
io = L[L.movement_type != "ADJUSTMENT"]
expected = np.where(io.movement_type == "IN", io.quantity, -io.quantity)
check("every IN/OUT step in running_balance equals +qty / -qty", (io.step == expected).all(), f"{(io.step == expected).sum()} of {len(io)}")
check("every ADJUSTMENT quantity is positive (5 to 50) in the quantity column",
      (adj.quantity > 0).all() and adj.quantity.min() == 5 and adj.quantity.max() == 50, f"min {adj.quantity.min()}, max {adj.quantity.max()}")
pos = int((adj.step == adj.quantity).sum()); neg = int((adj.step == -adj.quantity).sum())
other = len(adj) - pos - neg
check("every ADJUSTMENT step in running_balance is exactly +qty or -qty", other == 0, f"+qty {pos}, -qty {neg}, other {other}")
sign_ev = pd.DataFrame({"measure": ["adjustment rows", "applied as +qty", "applied as -qty", "neither", "share applied as negative"],
                        "value": [len(adj), pos, neg, other, round(neg / len(adj), 4)]})
sign_ev.to_csv(OUT / "q12_adjustment_sign_evidence.csv", index=False)
sign_ev

PASS every IN/OUT step in running_balance equals +qty / -qty | 234776 of 234776
PASS every ADJUSTMENT quantity is positive (5 to 50) in the quantity column | min 5, max 50
PASS every ADJUSTMENT step in running_balance is exactly +qty or -qty | +qty 1227, -qty 1227, other 0


,measure,value
0,adjustment rows,2454.0
1,applied as +qty,1227.0
2,applied as -qty,1227.0
3,neither,0.0
4,share applied as negative,0.5


## 3. Replay the ledger under each reading of the sign
Replay per pair: `opening_stock + IN − OUT ± ADJUSTMENT`, compared with the ledger's last balance and with `inventory_master.current_stock`.

**Read this carefully:** the "sign read from `running_balance`" replay reproducing the ledger on 180 of 180 pairs is **true by construction**, because the sign of each adjustment is taken from the same balance. It shows the data are consistent, not that the sign reading is correct. The independent evidence is in section 2 (every adjustment step is exactly +qty or −qty, and every IN/OUT step, including the first step of each pair measured against `opening_stock`, matches its quantity) and in section 3b (the 14 gaps match the last movements, none of which is an adjustment). The "all positive" and "all negative" replays are the real test of the unsigned-quantity hypothesis, and both fail on all 180 pairs.

In [4]:
L["signed_qty"] = np.where(L.movement_type == "IN", L.quantity,
                   np.where(L.movement_type == "OUT", -L.quantity, np.where(L.step < 0, -L.quantity, L.quantity)))
S = L.pivot_table(index=K, columns="movement_type", values="quantity", aggfunc="sum", fill_value=0).reset_index()
signed_sum = L.groupby(K).signed_qty.sum().rename("signed_sum").reset_index()
R = I[K + ["opening_stock", "current_stock", "ledger_latest_balance"]].merge(S, on=K).merge(signed_sum, on=K)
R["replay_all_positive"] = R.opening_stock + R.IN - R.OUT + R.ADJUSTMENT
R["replay_all_negative"] = R.opening_stock + R.IN - R.OUT - R.ADJUSTMENT
R["replay_no_adjustment"] = R.opening_stock + R.IN - R.OUT
R["replay_sign_from_balance"] = R.opening_stock + R.signed_sum
rows = []
for c in ["replay_all_positive", "replay_all_negative", "replay_no_adjustment", "replay_sign_from_balance"]:
    rows.append({"replay": c, "pairs_equal_ledger_last_balance": int((R[c] == R.ledger_latest_balance).sum()),
                 "pairs_equal_inventory_master": int((R[c] == R.current_stock).sum())})
variants = pd.DataFrame(rows); variants.to_csv(OUT / "q12_replay_variants.csv", index=False)
print(variants.to_string(index=False))
check("replay with sign read from running_balance reproduces the ledger on 180 of 180 pairs (true by construction: a consistency check, not independent evidence)",
      variants.loc[variants.replay == "replay_sign_from_balance", "pairs_equal_ledger_last_balance"].iloc[0] == 180)
check("replay with all adjustments positive reproduces the ledger on 0 pairs",
      variants.loc[variants.replay == "replay_all_positive", "pairs_equal_ledger_last_balance"].iloc[0] == 0)
check("the signed replay matches inventory_master on 166 pairs; the same 14 differ",
      variants.loc[variants.replay == "replay_sign_from_balance", "pairs_equal_inventory_master"].iloc[0] == 166)
fb = L[first.values]
fb_io = fb[fb.movement_type != "ADJUSTMENT"]; fb_adj = fb[fb.movement_type == "ADJUSTMENT"]
io_ok = ((fb_io.opening_stock + np.where(fb_io.movement_type == "IN", fb_io.quantity, -fb_io.quantity)) == fb_io.running_balance).all()
adj_ok = ((fb_adj.running_balance - fb_adj.opening_stock).abs() == fb_adj.quantity).all()
check("openings reconcile: the first IN/OUT of each pair equals opening_stock ± qty, and a first adjustment moves the balance by exactly its qty",
      io_ok and adj_ok, f"{len(fb_io)} pairs start with IN/OUT, {len(fb_adj)} start with an adjustment")

                  replay  pairs_equal_ledger_last_balance  pairs_equal_inventory_master
     replay_all_positive                                0                             0
     replay_all_negative                                0                             0
    replay_no_adjustment                                1                             1
replay_sign_from_balance                              180                           166
PASS replay with sign read from running_balance reproduces the ledger on 180 of 180 pairs (true by construction: a consistency check, not independent evidence)
PASS replay with all adjustments positive reproduces the ledger on 0 pairs
PASS the signed replay matches inventory_master on 166 pairs; the same 14 differ
PASS openings reconcile: the first IN/OUT of each pair equals opening_stock ± qty, and a first adjustment moves the balance by exactly its qty | 167 pairs start with IN/OUT, 13 start with an adjustment


## 3b. What the 14 gaps actually are
If the sign were the cause, the gap would depend on the adjustments. Instead, test whether each gap equals the net effect of the **last one or two ledger movements**, as if `inventory_master` was written just before they posted.

In [5]:
recs = []
for _, r in mism.iterrows():
    g = L[(L.product_id == r.product_id) & (L.branch_id == r.branch_id)]
    matched_n = None; cum = 0
    for n in (1, 2, 3):
        cum += g.step.iloc[-n]
        if cum == -r.gap:
            matched_n = n; break
    tail = g.tail(matched_n if matched_n else 1)
    recs.append({"product_id": r.product_id, "branch_id": r.branch_id, "current_stock": r.current_stock,
                 "ledger_latest_balance": r.ledger_latest_balance, "gap": r.gap,
                 "last_n_movements_explaining_gap": matched_n,
                 "missing_movement_ids": ";".join(tail.movement_id),
                 "missing_types": ";".join(tail.movement_type),
                 "missing_reference_ids": ";".join(tail.reference_id),
                 "missing_dates": ";".join(tail.movement_date.dt.strftime("%Y-%m-%d")),
                 "net_of_missing_movements": int(tail.step.sum())})
E = pd.DataFrame(recs); E.to_csv(OUT / "q12_mismatch_explained.csv", index=False)
print(E[["product_id", "branch_id", "gap", "last_n_movements_explaining_gap", "missing_types", "missing_dates"]].to_string(index=False))
check("each of the 14 gaps equals minus the net of the last 1 or 2 ledger movements",
      E.last_n_movements_explaining_gap.isin([1, 2]).all() and (E.net_of_missing_movements == -E.gap).all(),
      E.last_n_movements_explaining_gap.value_counts().to_dict())
check("for all 14, current_stock equals the ledger balance just before those movements",
      all((L[(L.product_id == r.product_id) & (L.branch_id == r.branch_id)].running_balance == r.current_stock).any() for _, r in mism.iterrows()))
type_counts = E.missing_types.str.split(";").explode().value_counts().to_dict()
check("none of the missing movements is an ADJUSTMENT, so the 14-gap finding does not depend on sign recovery", "ADJUSTMENT" not in type_counts, type_counts)
print("types of the missing movements:", type_counts)

product_id branch_id  gap  last_n_movements_explaining_gap missing_types         missing_dates
      P002    CHN001 -297                                1            IN            2025-01-20
      P004    PUN001 -165                                1            IN            2025-01-18
      P008    AHM001  -36                                1            IN            2025-01-24
      P010    AHM001 -400                                2         IN;IN 2025-01-24;2025-01-24
      P021    DEL001 -205                                1            IN            2025-01-17
      P023    HYD001 -183                                1            IN            2025-01-15
      P025    HYD001 -515                                2         IN;IN 2025-01-13;2025-01-13
      P026    PUN001 -145                                1            IN            2025-01-14
      P026    HYD001 -198                                1            IN            2025-01-18
      P028    DEL001 -223                         

PASS for all 14, current_stock equals the ledger balance just before those movements
PASS none of the missing movements is an ADJUSTMENT, so the 14-gap finding does not depend on sign recovery | {'IN': 16, 'OUT': 1}
types of the missing movements: {'IN': 16, 'OUT': 1}


## 4. Is there a single snapshot cut-off date?
If `inventory_master` were a snapshot taken on one date, every movement after that date would be missing everywhere. Check the final movement dates of the 166 matching pairs against the missing dates of the 14.

In [6]:
last_dt = L.groupby(K).movement_date.max().rename("last_date").reset_index()
Z = I[K + ["gap"]].merge(last_dt, on=K)
Z["mismatch"] = Z.gap != 0
cut = Z.groupby("mismatch").last_date.agg(["min", "max", "count"]).reset_index()
cut.to_csv(OUT / "q12_cutoff_check.csv", index=False)
print(cut.to_string(index=False))
miss_dates = pd.to_datetime(E.missing_dates.str.split(";").explode())
later_matched = int((Z[~Z.mismatch].last_date > miss_dates.min()).sum())
print("matching pairs whose last movement is later than the earliest missing movement:", later_matched, "of", int((~Z.mismatch).sum()))
check("no single cut-off date: many matching pairs have final movements later than the earliest missing one",
      later_matched > 100, f"{later_matched} of 166")

 mismatch        min        max  count
    False 2024-12-31 2025-01-28    166
     True 2025-01-09 2025-01-24     14
matching pairs whose last movement is later than the earliest missing movement: 163 of 166
PASS no single cut-off date: many matching pairs have final movements later than the earliest missing one | 163 of 166


## 5. Charts

In [7]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
a = adj.copy(); a["applied"] = np.where(a.step > 0, "applied as +qty", "applied as -qty")
for lab, col in [("applied as +qty", "#1B6B66"), ("applied as -qty", "#C8791A")]:
    d = a[a.applied == lab]
    ax[0].scatter(d.quantity, d.step, s=8, alpha=.5, label=f"{lab} ({len(d):,})", color=col)
ax[0].axhline(0, color="#999", lw=.8)
ax[0].set_xlabel("ADJUSTMENT quantity (column value)"); ax[0].set_ylabel("Step in running_balance")
ax[0].set_title("The sign lives in running_balance, not in quantity"); ax[0].legend(frameon=False)
ax[1].scatter(E.gap, -E.net_of_missing_movements, s=60, color="#16213A")
lim = [min(E.gap.min(), 0) - 30, 30]
ax[1].plot(lim, lim, color="#999", lw=.8, ls="--")
ax[1].set_xlabel("Gap: current_stock − ledger balance"); ax[1].set_ylabel("Minus net of last 1–2 movements")
ax[1].set_title("All 14 gaps = the last 1–2 movements, not posted")
plt.tight_layout(); plt.savefig(CH / "q12_01_sign_and_gap_explained.png", dpi=150); plt.close()
check("chart written", (CH / "q12_01_sign_and_gap_explained.png").exists())

PASS chart written


## 6. Validation

In [8]:
V = pd.DataFrame(CHECKS)
V.to_csv(HERE / "validation_report_week05_q12.csv", index=False)
print(V.passed.sum(), "of", len(V), "checks pass")
assert V.passed.all(), V[~V.passed]

18 of 18 checks pass
